In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
 
from sklearn.model_selection import train_test_split
 
import tensorflow as tf
from tensorflow.keras.layers import Layer, Dense, LSTM
from tensorflow.keras.callbacks import Callback
from tensorflow.keras import Model

In [ ]:
# Unduh dataset dari Kaggle menggunakan KaggleHub 
path = kagglehub.dataset_download("sumanthvrao/daily-climate-time-series-data")
file_name = 'DailyDelhiClimateTrain.csv'
file_path = os.path.join(path, file_name)
 
# Load dataset yang sudah diunduh dari Kaggle 
df = pd.read_csv(file_path)
 
# Definisikan variabel dari dataset yang akan digunakan
temp = df['meantemp'].values
 
window_size = 60
batch_size = 100
shuffle_buffer = 1000
 
#Membuat window untuk dataset
def windowed_dataset(series, window_size, batch_size, shuffle_buffer):
    series = tf.expand_dims(series, axis=-1)
    ds = tf.data.Dataset.from_tensor_slices(series)
    ds = ds.window(window_size + 1, shift=1, drop_remainder=True)
    ds = ds.flat_map(lambda w: w.batch(window_size + 1))
    ds = ds.shuffle(shuffle_buffer)
    ds = ds.map(lambda w: (w[:-1], w[-1:]))
    return ds.batch(batch_size).prefetch(1)
 
train_set = windowed_dataset(temp, window_size, batch_size, shuffle_buffer)

In [ ]:
class ResidualLSTM(tf.keras.layers.Layer):
    def __init__(self, units):
        super(ResidualLSTM, self).__init__()
        self.lstm = tf.keras.layers.LSTM(units, return_sequences=True)
        self.bn = tf.keras.layers.BatchNormalization()
        self.dense = tf.keras.layers.Dense(units, activation="relu")
 
    def call(self, inputs):
        x = self.lstm(inputs)
        x = self.bn(x)
        residual = self.dense(inputs)
        return x + residual

In [ ]:
class ResidualLSTMModel(tf.keras.Model):
    def __init__(self):
        super(ResidualLSTMModel, self).__init__()
        self.residual_lstm1 = ResidualLSTM(60)
        self.residual_lstm2 = ResidualLSTM(60)
        self.dense1 = tf.keras.layers.Dense(30, activation="relu")
        self.dense2 = tf.keras.layers.Dense(10, activation="relu")
        self.dense3 = tf.keras.layers.Dense(1)
 
    def call(self, inputs):
        x = self.residual_lstm1(inputs)
        x = self.residual_lstm2(x)
        x = self.dense1(x)
        x = self.dense2(x)
        return self.dense3(x)

In [ ]:
class CustomHuberLoss(tf.keras.losses.Loss):
    def __init__(self, delta=1.0):
        super(CustomHuberLoss, self).__init__()
        self.delta = delta
 
    def call(self, y_true, y_pred):
        error = y_true - y_pred
        is_small_error = tf.abs(error) <= self.delta
        squared_loss = tf.square(error) / 2
        linear_loss = self.delta * (tf.abs(error) - self.delta / 2)
        return tf.where(is_small_error, squared_loss, linear_loss)  

In [ ]:
class AdaptiveLearningRateScheduler(Callback):
    def __init__(self, factor=0.5, patience=3, min_lr=1e-6, max_lr=1e-2):
        super(AdaptiveLearningRateScheduler, self).__init__()
        self.factor = factor
        self.patience = patience
        self.min_lr = min_lr
        self.max_lr = max_lr
        self.wait = 0
        self.best_loss = float('inf')
    
    def on_epoch_end(self, epoch, logs=None):
        current_loss = logs.get("loss")
        lr = tf.keras.backend.get_value(self.model.optimizer.learning_rate)
        
        if current_loss < self.best_loss:
            self.best_loss = current_loss
            self.wait = 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                new_lr = max(lr * self.factor, self.min_lr)
                self.model.optimizer.learning_rate.assign(new_lr)
                print(f"\nReducing learning rate to {new_lr}")
                self.wait = 0

In [ ]:
optimizer = tf.keras.optimizers.SGD(learning_rate=1.0000e-04, momentum=0.9)
 
model = ResidualLSTMModel
model.compile(loss=CustomHuberLoss(delta=0.5), optimizer=optimizer)
 
lr_callback = AdaptiveLearningRateScheduler()
model.fit(train_set, epochs=100, callbacks=[lr_callback])